[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/crewai.ipynb)

# Research assistant — CrewAI Flow

**Task.** Answer which interventions reduce household food waste using a five-stage CrewAI research flow.

**Read this notebook as:** choose a provider → load evidence → declare flow stages → plan, retrieve, synthesise and critique.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [1]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [2]:
import contextlib
import io
import json
import os
import tempfile
from pathlib import Path
from typing import ClassVar, Literal

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai.flow.flow import Flow, listen, start  # noqa: E402
from pydantic import BaseModel, ConfigDict, Field  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.safety import SafetyEngine, SafetyPolicy, UntrustedContent  # noqa: E402
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Crew roles and flow
Planner, evidence reviewer, writer and critic responsibilities are explicit methods. Passages are isolated as untrusted data; only validated source IDs enter the ledger and citations.

In [3]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})


def fresh_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def search(query):
    terms = set(query.casefold().split())
    return [
        r
        for r in catalogue["records"]
        if terms & set((r["title"] + " " + r["passage"]).casefold().split())
    ]


class ResearchFlow(Flow):
    @start()
    async def planner(self):
        self.client = fresh_model()
        plan = await propose(
            self.client,
            QueryPlan,
            "Plan searches for smaller plates, meal planning, and "
            "information-only campaigns; return at most four focused queries.",
        )
        return {"plan": plan, "trace": [{"event": "plan"}], "model_calls": 1}

    @listen(planner)
    def evidence_reviewer(self, state):
        retrieved = {r["source_id"]: r for q in state["plan"].queries for r in search(q)}
        safety = SafetyEngine(SafetyPolicy())
        assessments = {
            sid: safety.inspect_retrieved_content(
                UntrustedContent(source_id=sid, text=r["passage"])
            )
            for sid, r in retrieved.items()
        }
        safe = {sid: r for sid, r in retrieved.items() if not assessments[sid].indicators}
        return {
            **state,
            "safe": safe,
            "trace": [
                *state["trace"],
                {
                    "event": "retrieve",
                    "isolated": [sid for sid, a in assessments.items() if a.indicators],
                },
            ],
        }

    @listen(evidence_reviewer)
    async def claim_ledger(self, state):
        ledger = await propose(
            self.client,
            Ledger,
            f"Extract one source-grounded claim per record from {state['safe']}. "
            "Use supporting for reductions and conflicting for inconsistent effects.",
        )
        claims = tuple(c for c in ledger.claims if c.source_id in state["safe"])
        return {
            **state,
            "claims": claims,
            "model_calls": 2,
            "trace": [
                *state["trace"],
                {
                    "event": "ledger",
                    "conflicts": [c.source_id for c in claims if c.stance == "conflicting"],
                },
            ],
        }

    @listen(claim_ledger)
    async def writer(self, state):
        answer = await propose(
            self.client,
            Synthesis,
            f"Synthesise only {state['claims']} with conditions, "
            "conflicts and source IDs; use outcome qualified_answer.",
        )
        return {
            **state,
            "answer": answer,
            "model_calls": 3,
            "trace": [*state["trace"], {"event": "synthesis"}],
        }

    @listen(writer)
    async def critic(self, state):
        decision = await propose(
            self.client,
            CriticOutput,
            f"Accept only if supported, qualified and cited: {state['answer'].model_dump()}",
        )
        decision = CriticDecision(accepted=decision.accepted, issues=decision.issues)
        valid = set(state["answer"].source_ids).issubset(state["safe"])
        return {
            **state,
            "outcome": state["answer"].outcome if decision.accepted and valid else "abstention",
            "model_calls": 4,
            "termination": "criteria_met",
            "trace": [
                *state["trace"],
                {"event": "critic", "accepted": decision.accepted, "citations_valid": valid},
            ],
        }


async def run_flow():
    with contextlib.redirect_stdout(io.StringIO()):
        return await ResearchFlow().kickoff_async()


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_flow()
second = await run_flow() if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["answer"].source_ids,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "flow_stages": 5},
    "fallback": "invalid evidence yields abstention",
}

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResearchFlow                                                                                             │
│  ID: 03f1b36c-e8be-4a7c-beb0-b0b15310e1c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResearchFlow                                                                                             │
│  ID: 03f1b36c-e8be-4a7c-beb0-b0b15310e1c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: planner                                                                                                │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: planner                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: writer                                                                                                 │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: critic                                                                                                 │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: critic                                                                                                 │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResearchFlow                                                                                             │
│  ID: f7f97a31-fc62-4dad-80e3-d6ae46d41c59                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'evaluation': {'component': True,
  'trajectory': True,
  'task': True,
  'safety': True,
  'repeated_run': True},
 'trace': [{'event': 'plan'},
  {'event': 'retrieve', 'isolated': []},
  {'event': 'ledger', 'conflicts': ['fw-003']},
  {'event': 'synthesis'},
  {'event': 'critic', 'accepted': True, 'citations_valid': True}],
 'resource_report': {'model_calls': 4, 'flow_stages': 5},
 'fallback': 'invalid evidence yields abstention'}

## Limitation
Crew roles improve readability but add runtime and telemetry/storage concerns; deterministic mock runs do not measure real multi-agent disagreement.